# UK Used Car Price Prediction: 04. Modeling dan Evaluasi

Notebook ini membandingkan Ridge dengan beberapa konfigurasi XGBoost. Pemilihan model memakai MAE cross-validation pada data training saja; test set hanya dipakai untuk evaluasi akhir setelah konfigurasi dipilih.

**Input:** `data/processed/cars_clean.csv` dan `models/03_ridge_baseline.joblib` dari tahap 03.  
**Output:** `models/04_best_model.joblib`, `reports/modeling_cv_results.csv`, dan `reports/model_metrics.json`.

Catatan: notebook 03 sudah menampilkan skor baseline pada test set. Jangan gunakan skor test untuk memilih konfigurasi atau tuning; gunakan hasil cross-validation di notebook ini.

## 1. Setup

Jalankan notebook dengan working directory di folder `notebooks/`, sesuai notebook sebelumnya.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling import run_modeling

print("Project:", PROJECT_ROOT)

Project: C:\Users\muhgu\uk-used-car-price-prediction


## 2. Cross-validation dan pemilihan model

Tiga nilai alpha Ridge dan empat konfigurasi XGBoost dibandingkan dengan 3-fold cross-validation. Setiap fold mempelajari preprocessing dari bagian training fold-nya sendiri. Skor MAE dihitung dalam pound setelah prediksi dikembalikan dari skala log.

In [2]:
cv_results, test_metrics, metrics = run_modeling()

print("Ranking cross-validation (MAE lebih rendah lebih baik):")
display(cv_results)
print("Model terpilih:", metrics["selected_model"])
print(f"MAE cross-validation terbaik: £{metrics['best_cv_mae']:,.2f}")

Fitting 3 folds for each of 7 candidates, totalling 21 fits
Ranking cross-validation (MAE lebih rendah lebih baik):


,rank_test_score,model,model_parameters,mean_cv_mae,std_cv_mae
6,1,XGBRegressor,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.1...",1311.805664,6.448178
4,2,XGBRegressor,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...",1433.526733,6.051422
5,3,XGBRegressor,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.1...",1518.571859,13.635012
0,4,Ridge,"{""alpha"": 1.0, ""solver"": ""lsqr""}",1646.635019,12.614617
1,5,Ridge,"{""alpha"": 10.0, ""solver"": ""lsqr""}",1663.485935,14.831039
3,6,XGBRegressor,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.0...",1709.018473,11.123143
2,7,Ridge,"{""alpha"": 100.0, ""solver"": ""lsqr""}",1818.857960,18.847513


Model terpilih: {'name': 'XGBRegressor', 'parameters': {'n_estimators': 180, 'max_depth': 7, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0}}
MAE cross-validation terbaik: £1,311.81


## 3. Evaluasi akhir pada test set

Model terpilih ditentukan oleh cross-validation training saja. Tabel berikut membandingkan baseline median, Ridge tahap 03, dan model terpilih pada split test 20% yang sama. Metrik utama adalah MAE; RMSE dan R² menjadi pelengkap.

In [3]:
display(test_metrics.style.format({
    "MAE (£)": "£{:,.0f}",
    "RMSE (£)": "£{:,.0f}",
    "R²": "{:.3f}",
}))

,MAE (£),RMSE (£),R²
Selected model,"£1,319","£2,295",0.948
Ridge baseline,"£1,651","£3,090",0.905
Median dummy,"£6,854","£10,271",-0.051


## 4. Model dan metrik yang disimpan

Model terpilih tersimpan terpisah dari baseline agar hasil tahap 03 tetap tersedia. CV ranking dan metrik holdout disimpan untuk dokumentasi dan penggunaan berikutnya.

In [4]:
model_path = PROJECT_ROOT / "models" / "04_best_model.joblib"
cv_path = PROJECT_ROOT / "reports" / "modeling_cv_results.csv"
metrics_path = PROJECT_ROOT / "reports" / "model_metrics.json"
print("Model:", model_path)
print("CV results:", cv_path)
print("Metrics:", metrics_path)

Model: C:\Users\muhgu\uk-used-car-price-prediction\models\04_best_model.joblib
CV results: C:\Users\muhgu\uk-used-car-price-prediction\reports\modeling_cv_results.csv
Metrics: C:\Users\muhgu\uk-used-car-price-prediction\reports\model_metrics.json


## Menggunakan model terpilih

Estimator tersimpan menerima DataFrame dengan fitur input yang sama seperti data bersih, tanpa kolom target `price`. Pipeline di dalam model otomatis membuat `car_age` dan menerapkan preprocessing.

In [5]:
import joblib

best_model = joblib.load(model_path)
sample_cars = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "cars_clean.csv").drop(columns="price").head(5)
predicted_prices = best_model.predict(sample_cars)
pd.DataFrame({"predicted_price (£)": predicted_prices})

,predicted_price (£)
0,14134.614258
1,16934.921875
2,12836.109375
3,19645.593750
4,17904.152344
